# GStar Assignment 2 — Problem 2 (extra credit): GRPO on Leduc Hold'em poker (Kaggle, 2× T4)

**Question:** can RL with verifiable rewards teach Qwen3-1.7B to *reason* about an imperfect-information game, and
does the kind of reward matter? Each prompt is one decision in Leduc Hold'em; the model thinks in `<think>` and
answers `fold / check / call / bet / raise`. Rewards come from an exact equilibrium of the game (CFR+), so the model
can be graded perfectly: we measure **exploitability** (chips/hand a perfect adversary wins against it), chips/hand
against fixed bots, and decision quality on **held-out situations**. Design: `problem2/README.md`.

**First time: smoke test (about 20–30 min, interactive).** Set `SMOKE = True` and *Run All*. It should end with `done:`.

**Real runs** (GPU T4 ×2, Internet on, then *Save Version → Save & Run All (Commit)*), one per session:
- **Session A:** `REWARD = "ev"` → bots + zero-shot baseline (L512, L1024), then GRPO with the exact equilibrium-value reward (~5–6 h).
- **Session B:** `REWARD = "outcome"` → GRPO with the noisy "play the hand out" reward (~4–5 h).

Then download `outputs_p2_<REWARD>.zip` from each version's *Output* tab.


In [ ]:
import os, time, json, shutil, subprocess
SESSION_START = time.time()

SMOKE = False           # True = ~20-30 min pipeline test at tiny size (run this once first)
REWARD = "ev"           # "ev" (session A, also runs zero-shot) or "outcome" (session B)
RUN_ZEROSHOT = REWARD == "ev"
MAX_TOKENS = 512
REPO_URL = "https://github.com/huypg-zenai/GStar-Assignment-2-GRPO.git"
BRANCH = "main"
WANDB_ENTITY = "phunghuysoict-hust"
WANDB_PROJECT = "gstar-a2-grpo"   # Problem 2 runs are grouped as p2_leduc_*
SESSION_LIMIT_MIN = 12 * 60   # Kaggle GPU session limit
SAFETY_MIN = 25               # kept free for the final eval, report and zip
OUT = "/kaggle/working/outputs_p2"
REPO = "/kaggle/tmp/repo"     # outside /kaggle/working so the code isn't duplicated in the outputs
VENV = "/kaggle/tmp/venv"     # Python 3.12 env: Kaggle's Python 3.13 is too new for vLLM 0.8.5
PY = f"{VENV}/bin/python"
os.makedirs(OUT, exist_ok=True)
def minutes_left():
    return SESSION_LIMIT_MIN - (time.time() - SESSION_START) / 60
# W&B key from Kaggle Secrets (never hard-code it in the notebook)
WANDB_ARGS = ""
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    WANDB_ARGS = f"--wandb_project {WANDB_PROJECT} --wandb_entity {WANDB_ENTITY} --wandb_group p2_leduc_{REWARD}"
    print("W&B: on ->", WANDB_ENTITY + "/" + WANDB_PROJECT)
except Exception as e:
    print("W&B: off (no WANDB_API_KEY secret attached):", type(e).__name__)
print("Problem 2 Leduc | reward:", REWARD, "| zero-shot:", RUN_ZEROSHOT, "| smoke:", SMOKE)

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
n = len(subprocess.check_output(["nvidia-smi", "-L"]).decode().strip().splitlines())
print("GPUs:", n)
if n < 2:
    print("WARNING: expected 2 GPUs (GPU T4 x2). With one GPU, vLLM and the policy share it and memory is tight.")


In [ ]:
if os.path.exists(REPO):
    shutil.rmtree(REPO)
!git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
commit = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
print("commit", commit)
json.dump({"repo": REPO_URL, "commit": commit, "problem": 2, "reward": REWARD}, open(f"{OUT}/run_info.json", "w"))

In [ ]:
# ~3-6 min. Kaggle's default Python (3.13) is too new for vLLM 0.8.5 (needs <3.13), so everything runs in a
# separate Python 3.12 env made with uv. All scripts below use {PY}; the notebook kernel itself only runs light code.
!pip install -q uv 2>&1 | tail -1
!uv venv -q --python 3.12 --seed {VENV}
!uv pip install -q --python {PY} -r {REPO}/kaggle/requirements_t4.txt 2>&1 | tail -5
!{PY} -c "import sys, torch, vllm, transformers, peft; print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda ok', torch.cuda.is_available(), '| vllm', vllm.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)"


In [ ]:
# CPU tests: Leduc engine vs OpenSpiel reference numbers, equilibrium, rewards, prompts (+ Problem 1 tests)
!cd {REPO} && {PY} -m pytest -q tests/test_leduc.py tests/test_tasks.py
!cd {REPO} && {PY} problem2/leduc.py baselines


In [ ]:
# Baselines: scripted bots + Qwen3-1.7B zero-shot, 8 samples at each of the 288 decision points (session A only)
if RUN_ZEROSHOT:
    !cd {REPO} && {PY} problem2/grpo_leduc.py zeroshot --lengths 512 1024 {'--smoke' if SMOKE else ''} --out {OUT}/zeroshot {WANDB_ARGS} --wandb_name p2_zeroshot 2>&1 | grep -v "it/s" | tee {OUT}/zeroshot.log


In [ ]:
# GRPO training on Leduc decisions. Stops early if the time budget runs out, then does the final eval (k=8).
BUDGET = int(minutes_left() - SAFETY_MIN)
RUN_DIR = f"{OUT}/train_leduc_{REWARD}"
print(f"time budget for training: {BUDGET} min -> {RUN_DIR}")
!cd {REPO} && {PY} problem2/grpo_leduc.py train --reward {REWARD} --max_tokens {MAX_TOKENS} --time_budget_min {BUDGET} {'--smoke' if SMOKE else ''} --out {RUN_DIR} {WANDB_ARGS} --wandb_name p2_leduc_{REWARD} 2>&1 | grep -v "it/s" | tee {OUT}/train.log


In [ ]:
# Table + plots for this session (re-run report_leduc.py locally on both sessions' outputs for the combined table)
!cd {REPO} && {PY} problem2/report_leduc.py {OUT} --save {OUT}/report
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{OUT}/report/results.md").read()))
if os.path.exists(f"{OUT}/report/problem2_curves.png"):
    display(Image(f"{OUT}/report/problem2_curves.png"))


In [ ]:
# Example decisions from the final eval: best and worst choices (for the analysis section)
import glob
evals = sorted(glob.glob(f"{OUT}/train_leduc_*/eval_step*_samples.jsonl"))
if evals:
    rows = [json.loads(l) for l in open(evals[-1])]
    rows.sort(key=lambda x: x["regret"])
    for label, ex in [("BEST", rows[:2]), ("WORST", rows[-2:])]:
        for x in ex:
            print(f"=== {label} | {x['key']} | action {x['action']} | regret {x['regret']:.2f} chips | {x['tokens']} tokens")
            print(x["response"][-700:], "\n")


In [ ]:
shutil.make_archive(f"/kaggle/working/outputs_p2_{REWARD}", "zip", OUT)
print(f"saved /kaggle/working/outputs_p2_{REWARD}.zip; minutes used: {(time.time() - SESSION_START) / 60:.0f}")
